# LeNet实战 - 第一个卷积神经网络

本notebook实现和训练经典的LeNet-5网络:
- LeNet-5架构详解
- 从零实现LeNet
- 在Fashion-MNIST上训练
- 性能分析和改进
- 可视化卷积层学习的特征

LeNet是深度学习历史上的里程碑!

## 第一部分: LeNet-5架构

### 1.1 历史背景

**LeNet-5** (1998年):
- 作者: Yann LeCun (深度学习三巨头之一)
- 目的: 手写数字识别
- 应用: ATM机支票识别
- 意义: 第一个成功的CNN

### 1.2 网络结构

LeNet-5包含:
1. **卷积层1**: 6个5×5卷积核
2. **池化层1**: 2×2平均池化
3. **卷积层2**: 16个5×5卷积核
4. **池化层2**: 2×2平均池化
5. **全连接层1**: 120个神经元
6. **全连接层2**: 84个神经元
7. **输出层**: 10个神经元(10类数字)

```
输入(28×28) 
    ↓
Conv2d(1→6, kernel=5, padding=2) + Sigmoid
    ↓ (28×28×6)
AvgPool2d(kernel=2, stride=2)
    ↓ (14×14×6)
Conv2d(6→16, kernel=5) + Sigmoid
    ↓ (10×10×16)
AvgPool2d(kernel=2, stride=2)
    ↓ (5×5×16)
Flatten()
    ↓ (400)
Linear(400→120) + Sigmoid
    ↓ (120)
Linear(120→84) + Sigmoid
    ↓ (84)
Linear(84→10)
    ↓ (10)
输出
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
import time

---

## 第二部分: 实现LeNet-5

### 2.1 原始LeNet-5

In [ ]:
class LeNet(nn.Module):
    """经典LeNet-5网络"""
    def __init__(self):
        super().__init__()
        # 卷积层1: 1→6通道, 5×5卷积核
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, padding=2)
        # 池化层1: 2×2平均池化
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
        # 卷积层2: 6→16通道, 5×5卷积核
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        # 池化层2
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
        # 全连接层
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
    
    def forward(self, x):
        # Conv1 + Sigmoid + Pool1
        x = self.pool1(torch.sigmoid(self.conv1(x)))
        # Conv2 + Sigmoid + Pool2
        x = self.pool2(torch.sigmoid(self.conv2(x)))
        # 展平
        x = x.view(-1, 16 * 5 * 5)
        # 全连接层
        x = torch.sigmoid(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        x = self.fc3(x)
        return x

# 创建模型
net = LeNet()
print(net)

### 2.2 使用Sequential简化

In [ ]:
# 更简洁的实现
net = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5), nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(16 * 5 * 5, 120), nn.Sigmoid(),
    nn.Linear(120, 84), nn.Sigmoid(),
    nn.Linear(84, 10)
)

print(net)

### 2.3 验证输出形状

In [ ]:
# 输入: batch=1, channels=1, height=28, width=28
X = torch.rand(1, 1, 28, 28)

print("各层输出形状:")
print("-" * 50)
for i, layer in enumerate(net):
    X = layer(X)
    print(f"第{i}层 {layer.__class__.__name__:15s} 输出形状: {X.shape}")

print("\n最终输出: 10个类别的分数")

### 2.4 参数量统计

In [ ]:
def count_parameters(model):
    """统计模型参数"""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

total, trainable = count_parameters(net)
print(f"总参数量: {total:,}")
print(f"可训练参数: {trainable:,}")
print(f"模型大小: {total * 4 / 1024:.2f} KB (float32)")

# 逐层参数量
print("\n各层参数量:")
print("-" * 60)
for name, param in net.named_parameters():
    print(f"{name:20s}: {param.numel():>8,}  形状: {list(param.shape)}")

---

## 第三部分: 准备数据

使用Fashion-MNIST数据集(10类服装图片)

In [ ]:
# 数据转换
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # 归一化到[-1, 1]
])

# 下载数据
train_dataset = torchvision.datasets.FashionMNIST(
    root='../data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.FashionMNIST(
    root='../data', train=False, transform=transform, download=True)

# 数据加载器
batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"训练集大小: {len(train_dataset)}")
print(f"测试集大小: {len(test_dataset)}")
print(f"Batch数量: {len(train_loader)} (训练), {len(test_loader)} (测试)")

# 类别名称
classes = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

### 可视化样本

In [ ]:
# 显示部分训练样本
def show_images(images, labels, preds=None):
    """显示图像和标签"""
    n = len(images)
    fig, axes = plt.subplots(2, n//2, figsize=(12, 5))
    axes = axes.flatten()
    
    for i in range(n):
        img = images[i].squeeze().numpy()
        axes[i].imshow(img, cmap='gray')
        title = f'{classes[labels[i]]}'
        if preds is not None:
            title += f'\n预测: {classes[preds[i]]}'
        axes[i].set_title(title, fontsize=10)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

# 获取一个batch
images, labels = next(iter(train_loader))
show_images(images[:8], labels[:8])

---

## 第四部分: 训练模型

### 4.1 训练函数

In [ ]:
def train_epoch(net, train_loader, criterion, optimizer, device):
    """训练一个epoch"""
    net.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        
        # 前向传播
        pred = net(X)
        loss = criterion(pred, y)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 统计
        total_loss += loss.item() * X.size(0)
        correct += (pred.argmax(1) == y).sum().item()
        total += X.size(0)
    
    return total_loss / total, correct / total

def evaluate(net, test_loader, criterion, device):
    """评估模型"""
    net.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            pred = net(X)
            loss = criterion(pred, y)
            
            total_loss += loss.item() * X.size(0)
            correct += (pred.argmax(1) == y).sum().item()
            total += X.size(0)
    
    return total_loss / total, correct / total

def train(net, train_loader, test_loader, num_epochs, lr, device):
    """完整训练流程"""
    # 初始化参数
    def init_weights(m):
        if type(m) == nn.Linear or type(m) == nn.Conv2d:
            nn.init.xavier_uniform_(m.weight)
    
    net.apply(init_weights)
    net.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(net.parameters(), lr=lr)
    
    # 记录历史
    history = {
        'train_loss': [], 'train_acc': [],
        'test_loss': [], 'test_acc': []
    }
    
    print(f"训练设备: {device}")
    print("="*70)
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_epoch(net, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(net, test_loader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        
        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"  Test Loss:  {test_loss:.4f}, Test Acc:  {test_acc:.4f}")
    
    elapsed = time.time() - start_time
    print("="*70)
    print(f"训练完成! 用时: {elapsed:.2f}秒")
    print(f"最终测试准确率: {history['test_acc'][-1]:.4f}")
    
    return history

print("训练函数已定义")

### 4.2 开始训练

In [ ]:
# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 创建新模型
net = nn.Sequential(
    nn.Conv2d(1, 6, kernel_size=5, padding=2), nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Conv2d(6, 16, kernel_size=5), nn.Sigmoid(),
    nn.AvgPool2d(kernel_size=2, stride=2),
    nn.Flatten(),
    nn.Linear(16 * 5 * 5, 120), nn.Sigmoid(),
    nn.Linear(120, 84), nn.Sigmoid(),
    nn.Linear(84, 10)
)

# 训练
lr = 0.9
num_epochs = 10

history = train(net, train_loader, test_loader, num_epochs, lr, device)

### 4.3 可视化训练过程

In [ ]:
# 绘制训练曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 损失曲线
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['test_loss'], label='Test Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('损失曲线', fontsize=14)
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# 准确率曲线
axes[1].plot(history['train_acc'], label='Train Acc', linewidth=2)
axes[1].plot(history['test_acc'], label='Test Acc', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('准确率曲线', fontsize=14)
axes[1].legend(fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n最终结果:")
print(f"  训练准确率: {history['train_acc'][-1]:.4f}")
print(f"  测试准确率: {history['test_acc'][-1]:.4f}")

---

## 第五部分: 模型预测和可视化

### 5.1 预测样本

In [ ]:
# 获取测试样本
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)

# 预测
net.eval()
with torch.no_grad():
    outputs = net(images)
    preds = outputs.argmax(1)

# 显示结果
images_cpu = images.cpu()
labels_cpu = labels.cpu()
preds_cpu = preds.cpu()

show_images(images_cpu[:8], labels_cpu[:8], preds_cpu[:8])

### 5.2 混淆矩阵

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# 收集所有预测
all_preds = []
all_labels = []

net.eval()
with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        pred = net(X).argmax(1)
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(y.numpy())

# 计算混淆矩阵
cm = confusion_matrix(all_labels, all_preds)

# 可视化
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.xlabel('预测类别', fontsize=12)
plt.ylabel('真实类别', fontsize=12)
plt.title('混淆矩阵', fontsize=14)
plt.tight_layout()
plt.show()

# 每类准确率
print("\n各类别准确率:")
for i, cls in enumerate(classes):
    acc = cm[i, i] / cm[i].sum()
    print(f"  {cls:12s}: {acc:.4f}")

### 5.3 可视化卷积核

In [ ]:
# 获取第一层卷积核
conv1_weight = net[0].weight.data.cpu()

# 显示6个卷积核
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i in range(6):
    kernel = conv1_weight[i, 0].numpy()
    axes[i].imshow(kernel, cmap='coolwarm')
    axes[i].set_title(f'卷积核 {i+1}', fontsize=12)
    axes[i].axis('off')

plt.suptitle('第一层卷积核 (5×5)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("观察: 不同卷积核学习到不同的边缘和纹理特征")

### 5.4 可视化特征图

In [ ]:
# 提取中间层特征
def get_activation(name, activation):
    def hook(model, input, output):
        activation[name] = output.detach()
    return hook

# 注册hook
activation = {}
net[0].register_forward_hook(get_activation('conv1', activation))
net[3].register_forward_hook(get_activation('conv2', activation))

# 获取一个样本
img, label = test_dataset[0]
img_input = img.unsqueeze(0).to(device)

# 前向传播
with torch.no_grad():
    _ = net(img_input)

# 显示原图和特征图
fig = plt.figure(figsize=(15, 10))

# 原图
plt.subplot(3, 7, 1)
plt.imshow(img.squeeze(), cmap='gray')
plt.title('原图', fontsize=10)
plt.axis('off')

# Conv1特征图(6个)
conv1_output = activation['conv1'].cpu().squeeze()
for i in range(6):
    plt.subplot(3, 7, i+2)
    plt.imshow(conv1_output[i], cmap='viridis')
    plt.title(f'Conv1-{i+1}', fontsize=10)
    plt.axis('off')

# Conv2特征图(前14个)
conv2_output = activation['conv2'].cpu().squeeze()
for i in range(14):
    plt.subplot(3, 7, i+8)
    plt.imshow(conv2_output[i], cmap='viridis')
    plt.title(f'Conv2-{i+1}', fontsize=10)
    plt.axis('off')

plt.suptitle(f'特征图可视化 - {classes[label]}', fontsize=14)
plt.tight_layout()
plt.show()

print("观察:")
print("- Conv1提取低层特征(边缘、纹理)")
print("- Conv2提取高层特征(形状、部件)")
print("- 特征图逐层变小,特征逐层抽象")

---

## 第六部分: 改进LeNet

### 6.1 现代版LeNet

In [ ]:
# 使用现代技术改进
class ModernLeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # 用ReLU替换Sigmoid
            nn.Conv2d(1, 6, kernel_size=5, padding=2),
            nn.ReLU(),
            # 用MaxPool替换AvgPool
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(6, 16, kernel_size=5),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 5 * 5, 120),
            nn.ReLU(),
            nn.Dropout(0.5),  # 添加Dropout
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(84, 10)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 创建并训练改进版
modern_net = ModernLeNet().to(device)
print(modern_net)

print("\n改进:")
print("✓ Sigmoid → ReLU (缓解梯度消失)")
print("✓ AvgPool → MaxPool (更好的特征选择)")
print("✓ 添加Dropout (防止过拟合)")

In [ ]:
# 训练改进版
lr = 0.01  # 降低学习率(Adam优化器更敏感)
num_epochs = 10

# 使用Adam优化器
def init_weights(m):
    if type(m) == nn.Linear or type(m) == nn.Conv2d:
        nn.init.xavier_uniform_(m.weight)

modern_net.apply(init_weights)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(modern_net.parameters(), lr=lr)

history_modern = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

print("训练改进版LeNet...")
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(modern_net, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(modern_net, test_loader, criterion, device)
    
    history_modern['train_loss'].append(train_loss)
    history_modern['train_acc'].append(train_acc)
    history_modern['test_loss'].append(test_loss)
    history_modern['test_acc'].append(test_acc)
    
    if (epoch + 1) % 2 == 0:
        print(f"Epoch {epoch+1}: Test Acc = {test_acc:.4f}")

print(f"\n最终测试准确率: {history_modern['test_acc'][-1]:.4f}")

### 6.2 对比原始版和改进版

In [ ]:
# 对比两个版本
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history['test_acc'], label='原始LeNet', linewidth=2)
plt.plot(history_modern['test_acc'], label='改进LeNet', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('测试准确率对比', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
models = ['原始LeNet', '改进LeNet']
accuracies = [history['test_acc'][-1], history_modern['test_acc'][-1]]
bars = plt.bar(models, accuracies, color=['steelblue', 'coral'], alpha=0.7)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('最终准确率对比', fontsize=14)
plt.ylim([0.8, 0.95])
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.4f}', ha='center', fontsize=12)

plt.tight_layout()
plt.show()

print("\n结论:")
print(f"原始LeNet: {history['test_acc'][-1]:.4f}")
print(f"改进LeNet: {history_modern['test_acc'][-1]:.4f}")
improvement = history_modern['test_acc'][-1] - history['test_acc'][-1]
print(f"提升: {improvement:.4f} ({improvement*100:.2f}%)")

---

## 小结

### LeNet的重要性

1. **历史意义**:
   - 首个成功的CNN
   - 证明了CNN在视觉任务上的潜力
   - 奠定了现代CNN的基础架构

2. **核心思想**:
   - 卷积层提取特征
   - 池化层降采样
   - 全连接层分类

3. **现代改进**:
   - Sigmoid → ReLU
   - AvgPool → MaxPool
   - 添加BatchNorm和Dropout
   - 更深的网络(VGG, ResNet等)

### 性能分析

| 模型 | 参数量 | 测试准确率 | 训练时间 |
|------|--------|------------|----------|
| 原始LeNet | ~60K | ~83-85% | ~2分钟 |
| 改进LeNet | ~60K | ~87-89% | ~2分钟 |
| 全连接MLP | ~500K | ~86-88% | ~3分钟 |

**优势**: CNN比MLP参数少10倍,但性能相当甚至更好!

### 关键要点

1. **架构设计**:
   - 卷积层逐层增加通道数
   - 池化层逐层减小空间尺寸
   - 最后展平接全连接层

2. **参数初始化**: Xavier初始化很重要
3. **学习率**: SGD用大学习率(0.9), Adam用小学习率(0.01)
4. **正则化**: Dropout有效防止过拟合

## 练习

1. **更深网络**: 添加更多卷积层,观察性能变化
2. **更多通道**: 增加卷积核数量(6→32, 16→64)
3. **BatchNorm**: 在每个卷积层后添加BatchNorm
4. **数据增强**: 使用随机裁剪、旋转等增强数据
5. **迁移学习**: 在CIFAR-10上微调LeNet
6. **可视化**: 使用Grad-CAM可视化模型关注区域